# 🏆 Model Comparison V2 (Baseline vs Word2Vec vs DistilBERT)
Independent module to extract metrics and charts without retraining.

In [ ]:
# 1. Environment Setup and Library Imports
import os
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformers[torch] datasets evaluate accelerate umap-learn
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/Project 2/project-nlp-challenge'
else:
    BASE_PATH = '.'

import pandas as pd
import numpy as np
import torch
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
import evaluate


In [ ]:
# 2. Re-Load Test Data and Saved Models for Comparison
df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/cleaned_data.csv'))
df['full_text'] = df['full_text'].fillna('')
dataset_full = Dataset.from_pandas(df[['full_text', 'label']])
dataset_split = dataset_full.train_test_split(test_size=0.2, seed=42)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize_function(examples):
    return tokenizer(examples["full_text"], padding="max_length", truncation=True)

print("Re-tokenizing test dataset for extraction...")
tokenized_datasets = dataset_split.map(tokenize_function, batched=True)

# Load DistilBERT
final_model_path = os.path.join(BASE_PATH, 'models/distilbert_classifier')
print(f"Loading model from: {final_model_path}")
model = AutoModelForSequenceClassification.from_pretrained(final_model_path)

# Reinitialize simple Trainer just to run '.predict()'
trainer = Trainer(model=model)

print("\u2705 Ready for Tournament Comparison!")


## 6. Model Evaluation & Scientific Comparison
We evaluate the Transformer performance and compare it against the Phase 2 Semantic baseline using different visualization techniques.

### 6.1 Standard Metrics & Confusion Matrix
Baseline evaluation of the fine-tuned DistilBERT model.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Predictions on test set
print("🧪 Evaluating BERT on test data...")
predictions_output = trainer.predict(tokenized_datasets['test'])
predicted_labels = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

# 2. Classification Report
report = classification_report(true_labels, predicted_labels, target_names=['FAKE NEWS', 'REAL NEWS'])
print("\n📊 Classification Report:")
print(report)

# 3. Confusion Matrix with Indigo Transformer Color
cm = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap=sns.light_palette("#6366F1", as_cmap=True), 
            xticklabels=['FAKE NEWS', 'REAL NEWS'], 
            yticklabels=['FAKE NEWS', 'REAL NEWS'],
            cbar=False, annot_kws={"size": 16, "weight": "bold"})

plt.title('Confusion Matrix: Transformer Deep Learning', fontsize=16, fontweight='bold', pad=20, color='#1A202C')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.show()

print("\n🎯 Context to REAL NEWS: (BERT understands full semantic relations and context nuance)")
print("🚩 Context to FAKE NEWS: (BERT identifies structural misinformation and linguistic patterns)")

### 6.2 ROC Curve Tournament (BERT vs Word2Vec)
Comparison of prediction confidence and separability between models.

In [ ]:
import joblib
import torch
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# 1. Save the new Advanced Transformer Model to Drive
distilbert_path = os.path.join(BASE_PATH, 'models/distilbert_classifier')
trainer.save_model(distilbert_path)
print(f"✅ Transformer Model saved successfully to {distilbert_path}")

# 2. Load Historical Models for comparison (from 02.2 notebook)
print("Loading Baseline and Semantic models...")
nb_tfidf = joblib.load(os.path.join(BASE_PATH, 'models/nb_tfidf_classifier.joblib'))
tfidf_vec = joblib.load(os.path.join(BASE_PATH, 'models/vectorizer.joblib'))

lr_w2v = joblib.load(os.path.join(BASE_PATH, 'models/lr_word2vec_classifier.joblib'))
w2v_model = joblib.load(os.path.join(BASE_PATH, 'models/word2vec_model.joblib'))

def document_vector(doc, model):
    words = [w for w in str(doc).split() if w in model.wv.index_to_key]
    if not words: return np.zeros(model.vector_size)
    return np.mean(model.wv[words], axis=0)

# 3. Get Probabilities for All Three Models
print("Calculating probabilities for tournament...")

# Baseline (Naive Bayes + TF-IDF)
X_test_tfidf = tfidf_vec.transform(dataset_split['test']['full_text'])
nb_probs = nb_tfidf.predict_proba(X_test_tfidf)[:, 1]

# Semantic (Logistic Regression + Word2Vec)
X_test_w2v = np.vstack([document_vector(text, w2v_model) for text in dataset_split['test']['full_text']])
w2v_probs = lr_w2v.predict_proba(X_test_w2v)[:, 1]

# Advanced Transformer (DistilBERT - Faster to use predictions in memory)
bert_probs = torch.nn.functional.softmax(torch.tensor(predictions_output.predictions), dim=-1).numpy()[:, 1]

# 4. Calculate ROC Curves
fpr_nb, tpr_nb, _ = roc_curve(true_labels, nb_probs)
fpr_w, tpr_w, _ = roc_curve(true_labels, w2v_probs)
fpr_b, tpr_b, _ = roc_curve(true_labels, bert_probs)

# 5. Plot the Ultimate Tournament
plt.figure(figsize=(10, 6))

plt.plot(fpr_nb, tpr_nb, color='#F56565', lw=2, linestyle='-.', label=f'Baseline Naive Bayes (AUC = {auc(fpr_nb, tpr_nb):.4f})')
plt.plot(fpr_w, tpr_w, color='#00B5AD', lw=2, linestyle='--', label=f'Semantic Word2Vec (AUC = {auc(fpr_w, tpr_w):.4f})')
plt.plot(fpr_b, tpr_b, color='#6366F1', lw=3, label=f'Advanced DistilBERT (AUC = {auc(fpr_b, tpr_b):.4f})')
plt.plot([0, 1], [0, 1], color='#CBD5E0', linestyle=':')

plt.title('Final Model Tournament: Baseline vs Semantic vs Transformer', fontsize=16, fontweight='bold', color='#1A202C')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.show()

### High-Dimensional Feature Extraction
We extract embeddings from both models for dimensionality reduction analysis.

In [ ]:
# Sampling and extraction shared by PCA, t-SNE, and UMAP
sample_size = 1000
sample_indices = range(min(sample_size, len(tokenized_datasets['test'])))
bert_sample = tokenized_datasets['test'].select(sample_indices)

def get_cls_embeddings(batch):
    inputs = {k: torch.tensor(v).to(model.device) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
    with torch.no_grad():
        outputs = model.distilbert(**inputs)
        cls_state = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    return {'bert_embeddings': cls_state}

print("🧠 Extracting BERT CLS Embeddings...")
bert_sample = bert_sample.map(get_cls_embeddings, batched=True, batch_size=32)
emb_bert = np.array(bert_sample['bert_embeddings'])
labels_s = np.array(bert_sample['label'])

print("⚡ Extracting Word2Vec Avg Vectors...")
emb_w2v = np.vstack([document_vector(t, w2v_model) for t in dataset_split['test'].select(sample_indices)['full_text']])

def plot_comparison_modular(name, method_obj):
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    colors = ['#FF8D2D', '#00B5AD']
    
    red_w2v = method_obj.fit_transform(emb_w2v)
    for label in [0, 1]:
        mask = labels_s == label
        axes[0].scatter(red_w2v[mask, 0], red_w2v[mask, 1], c=colors[label], label='Fake' if label==0 else 'Real', alpha=0.5)
    axes[0].set_title(f'Word2Vec Space ({name})', fontsize=14, fontweight='bold')
    
    red_bert = method_obj.fit_transform(emb_bert)
    for label in [0, 1]:
        mask = labels_s == label
        axes[1].scatter(red_bert[mask, 0], red_bert[mask, 1], c=colors[label], label='Fake' if label==0 else 'Real', alpha=0.5)
    axes[1].set_title(f'DistilBERT space ({name})', fontsize=14, fontweight='bold', color='#6366F1')
    
    plt.suptitle(f'Latent Space Comparison: {name}', fontsize=18, fontweight='bold', y=1.02)
    plt.show()

### 6.3 PCA Comparison
Projecting the most significant variance dimensions.

In [ ]:
from sklearn.decomposition import PCA
plot_comparison_modular('PCA', PCA(n_components=2))

### 6.4 t-SNE Comparison
Visualizing local clusters and neighborhood structures.

In [ ]:
from sklearn.manifold import TSNE
plot_comparison_modular('t-SNE', TSNE(n_components=2, perplexity=30, random_state=42))

### 6.5 UMAP Comparison
Preserving global and local structure for better separability visualization.

In [ ]:
import umap.umap_ as umap
plot_comparison_modular('UMAP', umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42))